# Customer Review Auto-Sorter
### A beginner-friendly NLP automation project

**The story:** A shop gets thousands of customer reviews. Nobody can read them all. So we build a small program that reads them for us and:

| Step | What the program does | Plain-English idea |
|---|---|---|
| 1 | Creates 100,000 sample reviews | Our "large dataset" |
| 2 | Reads them in small pieces | Don't swallow the whole elephant at once |
| 3 | Decides if each review is happy or angry | **Sentiment analysis** (NLP) |
| 4 | Decides what it is about (delivery, price, quality, service) | **Keyword categorizing** (NLP) |
| 5 | Watches new reviews arriving live and raises an alarm for angry ones | **Automation + real-time** |

Install once in a terminal: `pip install pandas matplotlib vaderSentiment`

## Step 0: Load the tools

In [ ]:
import random, time
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

reader = SentimentIntensityAnalyzer()   # a ready-made "mood detector" for text
random.seed(1)                          # makes the random data the same every run

**What happened:** we imported `pandas` (tables), `matplotlib` (charts), and VADER, a free tool that gives text a mood score from **-1 (very angry)** to **+1 (very happy)**. No training needed.

## Step 1: Make a big pile of reviews

In [ ]:
topics = {
    "delivery": ["The delivery was {}", "Shipping was {}"],
    "price":    ["The price is {}", "The cost is {}"],
    "quality":  ["The product quality is {}", "Build quality feels {}"],
    "service":  ["Customer service was {}", "The service team was {}"],
}
good = ["excellent", "great", "fantastic", "wonderful", "amazing"]
bad  = ["terrible", "awful", "horrible", "disappointing", "very poor"]

def make_review():
    topic = random.choice(list(topics))
    mood = random.choice([good, bad])
    return random.choice(topics[topic]).format(random.choice(mood))

pd.DataFrame({"review": [make_review() for _ in range(100_000)]}).to_csv("reviews.csv", index=False)
print("Created reviews.csv with 100,000 reviews")
print(make_review()); print(make_review()); print(make_review())

**What happened:** `make_review()` picks a random topic (delivery, price, quality, service) and a random good or bad word, then builds a sentence like *"Shipping was awful"*. We make 100,000 of them and save them in a file called `reviews.csv`.

*Using your own data? Skip this step and put your file name in Step 2. It just needs a column called `review`.*

## Step 2 & 3: Read in small pieces and score the mood

In [ ]:
def mood(text):
    score = reader.polarity_scores(text)["compound"]   # -1 to +1
    if score >= 0.3:
        return "happy"
    if score <= -0.3:
        return "angry"
    return "neutral"

parts = []
for piece in pd.read_csv("reviews.csv", chunksize=20_000):   # 20,000 rows at a time
    piece["mood"] = piece["review"].map(mood)
    parts.append(piece)
    print(f"Processed {len(piece):,} reviews")

data = pd.concat(parts)
data.head()

**What happened, line by line:**

- `def mood(text):` makes a small helper that answers "is this review happy or angry?"
- `polarity_scores(text)["compound"]` gets the mood score from -1 to +1.
- `if score >= 0.3` -> **happy**, `if score <= -0.3` -> **angry**, anything in between -> **neutral**.
- `chunksize=20_000` tells pandas to hand us the file in pieces of 20,000 rows. A huge file never has to fit in memory all at once.
- `.map(mood)` runs our helper on every review and stores the answer in a new `mood` column.
- `pd.concat(parts)` glues the processed pieces back together.

## Step 4: Find out what each review is about

In [ ]:
keywords = {
    "delivery": ["delivery", "shipping"],
    "price":    ["price", "cost"],
    "quality":  ["quality", "build"],
    "service":  ["service"],
}

def topic_of(text):
    text = text.lower()
    for topic, words in keywords.items():
        if any(w in text for w in words):
            return topic
    return "other"

data["topic"] = data["review"].map(topic_of)

table = pd.crosstab(data["topic"], data["mood"])
print(table)
table.plot.bar(title="Reviews by topic and mood", color=[{"angry": "#d9534f", "neutral": "#999999", "happy": "#5cb85c"}[c] for c in table.columns])
plt.ylabel("Number of reviews"); plt.xticks(rotation=0); plt.show()

**What happened:**

- `keywords` is a simple list of "if you see these words, the topic is this".
- `topic_of(text)` checks each topic's words; the first match wins; no match -> `"other"`.
- `pd.crosstab` counts how many reviews fall in each topic + mood combination.
- The chart instantly shows *where* customers are unhappy, which is the whole point of the project.

## Step 5: Real-time automation, watch new reviews and raise alarms

In [ ]:
def new_reviews(n=5):
    # PRETEND live feed. In real life, replace this with an API, database or file check.
    return [make_review() for _ in range(n)]

urgent = []

for round_number in range(1, 6):                 # 5 checks (use "while True:" to run forever)
    print(f"--- Check {round_number} ---")
    for text in new_reviews():
        feeling = mood(text)
        if feeling == "angry":
            urgent.append({"review": text, "topic": topic_of(text)})
            print(f"  ALARM [{topic_of(text)}]: {text}")
    time.sleep(1)                                 # wait 1 second before checking again

pd.DataFrame(urgent).to_csv("urgent_reviews.csv", index=False)
print(f"\nSaved {len(urgent)} urgent reviews to urgent_reviews.csv")

**What happened:**

- `new_reviews()` pretends that 5 new reviews arrive each time. Later you can swap it for a real source (see below).
- The `for` loop is our "watcher": it checks, handles the new reviews, waits a second, and checks again.
- For every new review we run the **same** `mood()` and `topic_of()` helpers from before.
- If it is angry, we **print an alarm** and add it to the `urgent` list.
- At the end the angry reviews are saved to `urgent_reviews.csv`, a ready-made to-do list for the support team.

This is automation: no human reads anything, yet the right people get the angry reviews.

## What did you build?

```
reviews.csv  -->  read in pieces  -->  mood  -->  topic  -->  chart
                                                      \
new live reviews  ------------------>  mood + topic -->  ALARM + urgent_reviews.csv
```

## Make it your own
| Want to... | Change this |
|---|---|
| Use real reviews | Put your CSV in Step 2 (needs a `review` column) |
| Use a real live feed | Rewrite `new_reviews()` to call an API (e.g. `requests.get(...)`) |
| Get alerts in Slack or email | Replace the `print("ALARM...")` line with a webhook or email call |
| Run forever | Change `for round_number in range(1, 6):` to `while True:` |
| Add more topics | Add lines to the `keywords` dictionary |
| Understand harder text (sarcasm, other languages) | Replace VADER with a `transformers` sentiment model |